In [3]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# SHARED PROTEIN INTERACTION

# BioGrid Data
Downloaded from: https://downloads.thebiogrid.org/BioGRID/Release-Archive/BIOGRID-4.4.241/

In [1]:
def process_biogrid_ppi_data(data):
    # Filter interactions down to physical interactions between two human proteins
    hsapien = 9606
    biogrid_ppi = data[(data['Organism ID Interactor A']==hsapien) & (data['Organism ID Interactor B']==hsapien) &
                       (data['Experimental System Type']=='physical')]
    # Filter to columns of interest + rename
    biogrid_ppi = biogrid_ppi[['Entrez Gene Interactor A', 'Entrez Gene Interactor B', 'Experimental System']]
    biogrid_ppi = biogrid_ppi.rename(columns={'Entrez Gene Interactor A':'A1_entrez', 'Entrez Gene Interactor B':'A2_entrez',
                                              'Experimental System':'experimental_system'})
    biogrid_ppi = biogrid_ppi.drop_duplicates().reset_index(drop=True).astype({'A1_entrez':'int', 'A2_entrez':'int'})
    
    # Filter out self interactions
    biogrid_ppi = biogrid_ppi[biogrid_ppi.A1_entrez != biogrid_ppi.A2_entrez].reset_index(drop=True)
    print('N interactions:', biogrid_ppi.shape[0])
    
    # Sort interacting genes - to find unique interaction pairs
    biogrid_unique = pd.DataFrame(np.sort(biogrid_ppi[['A1_entrez','A2_entrez']], axis=1), 
                                  columns=['A1_entrez', 'A2_entrez']).drop_duplicates()
    assert(biogrid_unique.shape[0] <= biogrid_ppi.shape[0])

    print('N interactions after sorting for unique pairs:', biogrid_unique.shape[0])
    print('N genes in interaction map:', pd.concat([biogrid_unique.A1_entrez, biogrid_unique.A2_entrez]).nunique())
    
    return biogrid_unique

In [4]:
biogrid_raw = pd.read_csv('../database_files/BioGRID/BIOGRID-ALL-4.4.241.tab3.txt', sep='\t',  low_memory=False)
biogrid_raw[:1]
biogrid_unique = process_biogrid_ppi_data(biogrid_raw)
biogrid_genes = pd.concat([biogrid_unique.A1_entrez, biogrid_unique.A2_entrez]).unique()
display(biogrid_unique[:2])

N interactions: 993165
N interactions after sorting for unique pairs: 862452
N genes in interaction map: 19995


,A1_entrez,A2_entrez
0,2318,6416
1,88,84665


In [9]:
## Calculate shared and total protein-protein interactions for all gene pairs
def compute_ppi_summary_for_pairs(ppi, gene_pairs):
    
    # Make ppi df symmetrical for merging with paralog pairs (which are the unique list)
    ppi_symmetric = pd.concat([ppi, ppi.rename(columns={'A1_entrez':'A2_entrez','A2_entrez':'A1_entrez'})])
    ppi_symmetric = ppi_symmetric.reset_index(drop=True)
    assert(ppi_symmetric.shape[0] == ppi.shape[0]*2)
    
    # Merge with paralog pairs - label interaction if paralog pair is found in PPI df
    gene_ppi = pd.merge(gene_pairs[['A1_entrez','A2_entrez']], ppi_symmetric, how='left', indicator='interact')
    gene_ppi.interact = gene_ppi.interact=='both'
    assert(gene_ppi.shape[0] == gene_pairs.shape[0])
    print('N. gene pairs that interact:', sum(gene_ppi.interact))
    
    # Use symmetric version of ppi table to get set of all interactors for each gene
    ppi_per_gene = ppi_symmetric.groupby('A1_entrez').agg({'A2_entrez':set}).reset_index()\
                                .rename(columns={'A1_entrez':'gene', 'A2_entrez':'ppi'})
    assert(ppi_per_gene.shape[0] == pd.concat([ppi.A1_entrez, ppi.A2_entrez]).nunique())

    # Merge ppi_per_gene with each A1 and A2 in all pairs
    # Note: pairs can have shared interactors even if there is no evidence they interact themselves
    df = pd.merge(gene_ppi, ppi_per_gene.rename(columns={'gene':'A1_entrez','ppi':'A1_ppi'}), how='left')
    df = pd.merge(df, ppi_per_gene.rename(columns={'gene':'A2_entrez','ppi':'A2_ppi'}), how='left')
    assert(df.shape[0] == gene_pairs.shape[0])
    print('N paralog pairs w/ 1+ interactor (A1 and/or A2):', df[(~df.A1_ppi.isna()) | (~df.A2_ppi.isna())].shape[0])
    
    # Fill NaNs with empty sets
    df['A1_ppi'] = df['A1_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    df['A2_ppi'] = df['A2_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    
    # Remove A2 gene in the set of interactors for A1 gene (and vice versa)
    # Don't want to include these in union for other calculations
    df.A1_ppi = df.apply(lambda x: x.A1_ppi - {x.A2_entrez}, axis=1)
    df.A2_ppi = df.apply(lambda x: x.A2_ppi - {x.A1_entrez}, axis=1)

    # Calculate total num interactors + shared interactors
    df['n_A1_ppi'] = df.apply(lambda x: len(x.A1_ppi), axis=1)
    df['n_A2_ppi'] = df.apply(lambda x: len(x.A2_ppi), axis=1)
    df['shared_ppi'] = df.apply(lambda x: x.A1_ppi.intersection(x.A2_ppi), axis=1)
    df['n_total_ppi'] = df.apply(lambda x: len(x.A1_ppi.union(x.A2_ppi)), axis=1)
    df['n_shared_ppi'] = df.apply(lambda x: len(x.shared_ppi), axis=1)

    # Calculate jaccard index for shared interactors
    def calc_jaccard_index(x):
        if x.n_shared_ppi == 0: return 0
        return x.n_shared_ppi / ((x.n_A1_ppi + x.n_A2_ppi) - x.n_shared_ppi)

    df['shared_ppi_jaccard_idx'] = df.apply(calc_jaccard_index, axis=1)

    # Calculate FET for overlap of interactors, N = all genes involved in interactions
    N = len(pd.concat([ppi.A1_entrez, ppi.A2_entrez]).unique())
    print('N genes involded in interactions:', N)
    assert(ppi_per_gene.shape[0] == N)
    # ctab:   | A2      | Not A2
    #      A1 | shared  | A1 only
    #  Not A1 | A2 only | N - union(A1, A2)
    def calc_fet_shared_ppi(x):
        ctab = pd.DataFrame({'A2': [x.n_shared_ppi, x.n_A2_ppi - x.n_shared_ppi],
                            'NA2': [x.n_A1_ppi - x.n_shared_ppi, N - x.n_total_ppi]}, index=['A1', 'NA1'])
        (OR, pval) = stats.fisher_exact(ctab)
        if pval==0: # Use smallest float64 number, to apply log10
            pval = np.nextafter(0, 1)
        log_pval = (-np.log10(pval)) if pval != 1 else 0
        log_pval = -log_pval if OR < 1 else log_pval # return negative pval if signif depletion in overlap
        return log_pval

    df['fet_ppi_overlap'] = df.apply(calc_fet_shared_ppi, axis=1)
    
    df = df.drop(columns=['A1_ppi','A2_ppi','n_A1_ppi','n_A2_ppi'])
    
    return df

In [16]:
import re

main_file_csv = pd.read_csv(
    '../results/CNA_quantitative_cluster_delta/potent_synthetic_lethal_hits_with_HGNC.csv',
    low_memory=False
)

# Keep only columns we actually have
cols_to_keep = [
    'Biomarker', 'Biomarker_HGNC',
    'BiomarkerCluster', 'BiomarkerCluster_HGNC',
    'TargetGene', 'TargetGene_HGNC'
]
df_clusters = main_file_csv[cols_to_keep].copy()

# Expand BiomarkerCluster into individual biomarker candidates
df_expanded = (
    df_clusters
    .assign(Biomarker_in_cluster=df_clusters['BiomarkerCluster'].str.split(r',\s*'))
    .explode('Biomarker_in_cluster')
)

# Map biomarker IDs in cluster to HGNC names for ORF detection
def map_symbol_from_cluster(row):
    ids = [x.strip() for x in row['BiomarkerCluster'].split(',')]
    syms = [x.strip() for x in row['BiomarkerCluster_HGNC'].split(',')]
    return dict(zip(ids, syms)).get(str(row['Biomarker_in_cluster']), None)

df_expanded['Biomarker_in_cluster_HGNC'] = df_expanded.apply(map_symbol_from_cluster, axis=1)

# Flag ORFs (case-insensitive match for C<number>orf<number>)
df_expanded['is_orf'] = df_expanded['Biomarker_in_cluster_HGNC'].str.match(r'^C\d+orf\d+$', case=False, na=False)

# Ensure integer IDs for PPI calculation
df_expanded['A1_entrez'] = df_expanded['Biomarker_in_cluster'].astype(int)
df_expanded['A2_entrez'] = df_expanded['TargetGene'].astype(int)

# Run PPI summary
ppi_results = compute_ppi_summary_for_pairs(
    biogrid_unique,
    df_expanded[['A1_entrez', 'A2_entrez']]
)

# Merge back cluster info and ORF flag
ppi_results = df_expanded.merge(ppi_results, on=['A1_entrez', 'A2_entrez'], how='left')

# Rank: interact → is_orf(False first) → n_shared_ppi → fet_ppi_overlap
representatives = (
    ppi_results
    .sort_values(
        ['BiomarkerCluster', 'TargetGene', 'interact', 'is_orf', 'n_shared_ppi', 'fet_ppi_overlap'],
        ascending=[True, True, False, True, False, False]
    )
    .groupby(['BiomarkerCluster', 'TargetGene'], as_index=False)
    .first()
)

# Rename Biomarker_in_cluster → Rep_Biomarker
representatives = representatives.rename(columns={'Biomarker_in_cluster': 'Rep_Biomarker'})

# Add Rep_Biomarker_HGNC from earlier mapping
representatives['Rep_Biomarker_HGNC'] = representatives['Biomarker_in_cluster_HGNC']

# Drop internal columns
representatives = representatives.drop(columns=['is_orf', 'Biomarker_in_cluster_HGNC'])

# Reorder columns
ordered_cols = [
    'Biomarker', 'Biomarker_HGNC',
    'BiomarkerCluster', 'BiomarkerCluster_HGNC',
    'Rep_Biomarker', 'Rep_Biomarker_HGNC',
    'TargetGene', 'TargetGene_HGNC',
    'A1_entrez', 'A2_entrez', 'interact', 'shared_ppi',
    'n_total_ppi', 'n_shared_ppi', 'shared_ppi_jaccard_idx',
    'fet_ppi_overlap'
]
representatives = representatives[ordered_cols]

# Save
representatives.to_csv(
    '../results/CNA_quantitative_cluster_delta/fet_ppi_overlap_biomarker_query_representative.csv',
    index=False
)

representatives

N. gene pairs that interact: 31
N paralog pairs w/ 1+ interactor (A1 and/or A2): 1304
N genes involded in interactions: 19995


,Biomarker,Biomarker_HGNC,BiomarkerCluster,BiomarkerCluster_HGNC,Rep_Biomarker,Rep_Biomarker_HGNC,TargetGene,TargetGene_HGNC,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,1004,CDH6,"1004, 55322","CDH6, C5orf22",1004,CDH6,7328,UBE2H,1004,7328,False,{},1961,0,0.000000,0.000000
1,1004,CDH6,"1004, 55322","CDH6, C5orf22",1004,CDH6,7849,PAX8,1004,7849,False,{},107,0,0.000000,0.000000
2,1004,CDH6,"1004, 55322","CDH6, C5orf22",1004,CDH6,8467,SMARCA5,1004,8467,False,{1499},395,1,0.002532,1.408685
3,10051,SMC4,"10051, 3840, 51068, 8706","SMC4, KPNA4, NMD3, B3GALNT1",10051,SMC4,4605,MYBL2,10051,4605,False,"{4609, 1025, 9219, 899, 142, 112399, 8467, 105...",392,19,0.048469,13.237624
4,10051,SMC4,"10051, 3840, 51068, 8706","SMC4, KPNA4, NMD3, B3GALNT1",10051,SMC4,54468,MIOS,10051,54468,False,"{57664, 8452, 25831, 8878, 112399, 4914, 2099,...",329,10,0.030395,7.540668
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
730,9846,GAB2,9846,GAB2,9846,GAB2,253260,RICTOR,9846,253260,False,"{3329, 3845, 79109, 3728, 51474, 4893, 1828, 5...",810,28,0.034568,14.959199
731,9846,GAB2,9846,GAB2,9846,GAB2,286053,NSMCE2,9846,286053,False,{7533},156,1,0.006410,0.685484
732,9846,GAB2,9846,GAB2,9846,GAB2,503834,DPRX,9846,503834,False,{},135,0,0.000000,0.000000
733,9847,C2CD5,9847,C2CD5,9847,C2CD5,4605,MYBL2,9847,4605,False,"{1025, 899, 6117, 2033, 1017}",225,5,0.022222,3.377627


In [17]:
import pandas as pd

# Load the representatives file we just saved
ppi_file = "../results/CNA_quantitative_cluster_delta/fet_ppi_overlap_biomarker_query_representative.csv"
representatives = pd.read_csv(ppi_file)

# Load the HGNC mapping file
hgnc_df = pd.read_csv("../database_files/gene_with_protein_product.txt", sep="\t")
entrez_to_symbol = dict(zip(hgnc_df["entrez_id"].astype(str), hgnc_df["symbol"]))

# Function to map Entrez IDs in shared_ppi to HGNC symbols
def map_shared_ppi_to_hgnc(val):
    if pd.isnull(val) or val == "set()":
        return ""
    # Clean and split the set string into individual IDs
    ids = [x.strip() for x in val.strip("{}").split(",") if x.strip()]
    symbols = [entrez_to_symbol.get(x, x) for x in ids]
    return ", ".join(symbols)

# Compute mapped HGNC list
shared_ppi_hgnc = representatives["shared_ppi"].apply(map_shared_ppi_to_hgnc)

# Insert new column immediately after shared_ppi
col_idx = representatives.columns.get_loc("shared_ppi") + 1
representatives.insert(col_idx, "shared_ppi_hgnc", shared_ppi_hgnc)

# Save, overwriting the PPI representatives file
representatives.to_csv(ppi_file, index=False)

representatives.head()

/var/folders/09/0r9l07110lg5nj9ndgsx9kl80000gn/T/ipykernel_75304/4087288859.py:8: DtypeWarning: Columns (35,38,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc_df = pd.read_csv("../database_files/gene_with_protein_product.txt", sep="\t")


,Biomarker,Biomarker_HGNC,BiomarkerCluster,BiomarkerCluster_HGNC,Rep_Biomarker,Rep_Biomarker_HGNC,TargetGene,TargetGene_HGNC,A1_entrez,A2_entrez,interact,shared_ppi,shared_ppi_hgnc,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,1004,CDH6,"1004, 55322","CDH6, C5orf22",1004,CDH6,7328,UBE2H,1004,7328,False,set(),,1961,0,0.000000,0.000000
1,1004,CDH6,"1004, 55322","CDH6, C5orf22",1004,CDH6,7849,PAX8,1004,7849,False,set(),,107,0,0.000000,0.000000
2,1004,CDH6,"1004, 55322","CDH6, C5orf22",1004,CDH6,8467,SMARCA5,1004,8467,False,{1499},CTNNB1,395,1,0.002532,1.408685
3,10051,SMC4,"10051, 3840, 51068, 8706","SMC4, KPNA4, NMD3, B3GALNT1",10051,SMC4,4605,MYBL2,10051,4605,False,"{4609, 1025, 9219, 899, 142, 112399, 8467, 105...","MYC, CDK9, MTA2, CCNF, PARP1, EGLN3, SMARCA5, ...",392,19,0.048469,13.237624
4,10051,SMC4,"10051, 3840, 51068, 8706","SMC4, KPNA4, NMD3, B3GALNT1",10051,SMC4,54468,MIOS,10051,54468,False,"{57664, 8452, 25831, 8878, 112399, 4914, 2099,...","PLEKHA4, CUL3, HECTD1, SQSTM1, EGLN3, NTRK1, E...",329,10,0.030395,7.540668


In [1]:
import pandas as pd

# File paths
potent_hits_file = "../results/CNA_quantitative_cluster_delta/potent_synthetic_lethal_hits_with_HGNC.csv"
ppi_file = "../results/CNA_quantitative_cluster_delta/fet_ppi_overlap_biomarker_query_representative.csv"
output_file = potent_hits_file.replace(".csv", "_ppi_validated.csv")

# Load data
potent_hits = pd.read_csv(potent_hits_file)
ppi_reps = pd.read_csv(ppi_file)

# Merge on BiomarkerCluster and TargetGene to align representative info
merged = potent_hits.merge(
    ppi_reps[
        ["BiomarkerCluster", "TargetGene", "Rep_Biomarker", "Rep_Biomarker_HGNC"]
    ],
    on=["BiomarkerCluster", "TargetGene"],
    how="left"
)

# Replace biomarker columns with representative biomarker info
merged["Biomarker"] = merged["Rep_Biomarker"]
merged["Biomarker_HGNC"] = merged["Rep_Biomarker_HGNC"]

# Recompute OncogeneAddiction: True if biomarker ID matches target ID
merged["OncogeneAddiction"] = merged["Biomarker"] == merged["TargetGene"]

# Drop the helper columns
merged = merged.drop(columns=["Rep_Biomarker", "Rep_Biomarker_HGNC"])

# Save updated file
merged.to_csv(output_file, index=False)

print(f"Updated potent hits saved to: {output_file}")

Updated potent hits saved to: ../results/CNA_quantitative_cluster_delta/potent_synthetic_lethal_hits_with_HGNC_ppi_validated.csv
